# Redrob Candidate Discovery & Ranking Pipeline (Standalone Google Colab)

This notebook runs the candidate ranking pipeline end-to-end entirely in the Colab runtime without requiring any Google Drive access or mount permissions. It clones the repository, downloads dependencies and models, and runs the ranking in under 5 minutes.

### Step 1: Clone the GitHub Repository
We clone the codebase directly into the local ephemeral directory.

In [ ]:
# Clone the repository (replace this URL with your final repo link)
!git clone https://github.com/umar/redrob-ai-challenge.git
%cd redrob-ai-challenge/sandbox

### Step 2: Install Python Libraries
Install the required libraries for indexing, ONNX runtime, and local execution.

In [ ]:
!pip install sentence-transformers faiss-cpu onnxruntime pandas numpy scipy transformers llama-cpp-python

### Step 3: Fetch the Qwen LLM Model
We download the `Qwen2.5-0.5B-Instruct` GGUF model directly from the official Hugging Face repository.

In [ ]:
!mkdir -p models
!huggingface-cli download Qwen/Qwen2.5-0.5B-Instruct-GGUF qwen2.5-0.5b-instruct-q4_k_m.gguf --local-dir models --local-dir-use-symlinks False

### Step 4: Fetch standard Reranker Models
We download the `ms-marco-MiniLM-L-12-v2` cross-encoder model weights from Hugging Face into our `models/` directory.

In [ ]:
!huggingface-cli download cross-encoder/ms-marco-MiniLM-L-12-v2 --local-dir models/ms-marco-MiniLM-L-12-v2 --local-dir-use-symlinks False

### Step 5: Verify/Download Precalculated Index Files
Ensure the index files (`features_v3.parquet`, `faiss_qwen_index.bin`, `splade_combined_v3.npz`, and the precomputed JD vectors) are located under `index/`.

*(Note: If these files were not pushed to your GitHub repo due to file size limits, you can download them from a public release or Hugging Face repository using `wget` or `huggingface-cli` below)*

In [ ]:
# Example to download the FAISS index if it's hosted on a Hugging Face Dataset/Model repo:
# !huggingface-cli download <YOUR_HF_USERNAME>/<YOUR_DATASET_REPO> faiss_qwen_index.bin --local-dir index --local-dir-use-symlinks False

### Step 6: Setup Linux llama-server Executables
Download and compile/extract the native Linux x64 build of `llama-server` on-the-fly.

In [ ]:
import urllib.request
import json
import zipfile
import os

print("Fetching latest llama.cpp Linux release info...")
req = urllib.request.Request("https://api.github.com/repos/ggerganov/llama.cpp/releases/latest")
req.add_header('User-Agent', 'Mozilla/5.0')
with urllib.request.urlopen(req) as response:
    data = json.loads(response.read().decode())

download_url = None
for asset in data['assets']:
    if "bin-ubuntu-x64.zip" in asset['name']:
        download_url = asset['browser_download_url']
        break

if not download_url:
    print("Error: Could not find Ubuntu x64 release!")
else:
    zip_path = "llama-linux-temp.zip"
    print(f"Downloading from {download_url}...")
    urllib.request.urlretrieve(download_url, zip_path)
    
    os.makedirs("bin/llama", exist_ok=True)
    print("Extracting Linux llama-server and shared library files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.namelist():
            if member.endswith(".so") or "llama-server" in member:
                filename = os.path.basename(member)
                if filename:
                    target_path = os.path.join("bin/llama", filename)
                    with zip_ref.open(member) as source, open(target_path, "wb") as target:
                        target.write(source.read())
                    print(f"  Extracted: {filename}")
    
    os.remove(zip_path)
    !chmod +x bin/llama/llama-server
    print("Linux llama-server ready!")

### Step 7: Run Pipeline & Validate Format
We start the background server, execute the python ranker, and automatically check the formatted outputs.

In [ ]:
import subprocess
import time
import socket

# Start llama-server in the background (using 2 cores / 4 threads as available on Colab)
llama_cmd = [
    "bin/llama/llama-server",
    "-m", "models/qwen2.5-0.5b-instruct-q4_k_m.gguf",
    "-c", "16384",
    "-np", "4",
    "--threads", "4",
    "--port", "8080"
]
print("Starting llama-server in the background...")
log_file = open("llama-server.log", "w")
err_file = open("llama-server-err.log", "w")
llama_proc = subprocess.Popen(llama_cmd, stdout=log_file, stderr=err_file)

# Wait for server to start
ready = False
for i in range(30):
    time.sleep(1)
    try:
        s = socket.create_connection(("127.0.0.1", 8080), timeout=1)
        s.close()
        ready = True
        break
    except:
        pass

if not ready:
    print("Error: llama-server failed to start! Check llama-server-err.log")
    llama_proc.terminate()
    llama_proc.wait()
    log_file.close()
    err_file.close()
else:
    print("Server is ready! Running the pipeline...")
    try: 
        # Execute the main pipeline script
        !python -u src/optimize_rank.py
        
        # Run validation checker to confirm format correctness
        print("\n--- RUNNING SUBMISSION VALIDATOR ---")
        !python validate_submission.py submission.csv
    finally:
        print("Pipeline finished. Terminating llama-server...")
        llama_proc.terminate()
        llama_proc.wait()
        log_file.close()
        err_file.close()
        print("Done.")